# Visualizations & Dashboards

Create, render, and publish data visualizations from notebooks. Combine them into drag-and-drop dashboards for real-time monitoring.

OpenModelStudio supports **9 visualization backends** with a unified `render()` abstraction.

| Backend | Output | Description |
|---------|--------|-------------|
| **matplotlib** | SVG | Standard Python plotting |
| **seaborn** | SVG | Statistical visualization |
| **plotly** | JSON | Interactive charts (zoom, pan, hover) |
| **bokeh** | JSON | Interactive streaming charts |
| **altair** | JSON | Declarative (Vega-Lite) |
| **plotnine** | SVG | ggplot2-style grammar of graphics |
| **datashader** | PNG | Server-side for millions of points |
| **networkx** | SVG | Network/graph visualizations |
| **geopandas** | SVG | Geospatial maps |

---
## 1. Matplotlib (SVG)

In [ ]:
import openmodelstudio as oms
import matplotlib.pyplot as plt
import numpy as np

# 1. Create a visualization record
viz = oms.create_visualization("training-loss",
    backend="matplotlib",
    description="Training loss over epochs")

# 2. Render it
fig, ax = plt.subplots()
epochs = np.arange(1, 21)
loss = 0.9 * np.exp(-0.15 * epochs) + 0.05
ax.plot(epochs, loss, color="#8b5cf6", linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training Loss")

# 3. Push rendered output to platform
output = oms.render(fig, viz_id=viz["id"])  # auto-detects matplotlib → SVG
oms.publish_visualization(viz["id"])
print("Matplotlib visualization published")

---
## 2. Plotly (Interactive JSON)

In [ ]:
import openmodelstudio as oms
import plotly.graph_objects as go

viz = oms.create_visualization("accuracy-curve",
    backend="plotly",
    description="Model accuracy vs epoch")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(1, 11)),
    y=[0.5, 0.62, 0.71, 0.78, 0.82, 0.85, 0.87, 0.89, 0.90, 0.91],
    mode="lines+markers",
    name="Accuracy",
    line=dict(color="#10b981"),
))
fig.update_layout(title="Model Accuracy", xaxis_title="Epoch", yaxis_title="Accuracy")

output = oms.render(fig, viz_id=viz["id"])  # auto-detects plotly → JSON
oms.publish_visualization(viz["id"])
print("Plotly visualization published")

---
## 3. Altair / Vega-Lite (Declarative)

In [ ]:
import openmodelstudio as oms
import altair as alt
import pandas as pd

viz = oms.create_visualization("feature-distribution",
    backend="altair",
    description="Distribution of model features")

data = pd.DataFrame({
    "feature": ["Age", "Fare", "Pclass", "SibSp", "Parch"],
    "importance": [0.28, 0.25, 0.22, 0.15, 0.10],
})

chart = alt.Chart(data).mark_bar(cornerRadiusTopLeft=3, cornerRadiusTopRight=3).encode(
    x=alt.X("feature", sort="-y"),
    y="importance",
    color=alt.Color("feature", scale=alt.Scale(scheme="category10")),
)

output = oms.render(chart, viz_id=viz["id"])  # auto-detects altair → Vega-Lite JSON
oms.publish_visualization(viz["id"])
print("Altair visualization published")

---
## 4. Seaborn (Statistical)

In [ ]:
import openmodelstudio as oms
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

viz = oms.create_visualization("correlation-heatmap",
    backend="seaborn",
    description="Feature correlation matrix")

data = pd.DataFrame(np.random.randn(100, 5), columns=["A", "B", "C", "D", "E"])
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(data.corr(), annot=True, cmap="coolwarm", ax=ax)

output = oms.render(fig, viz_id=viz["id"])
oms.publish_visualization(viz["id"])
print("Seaborn heatmap published")

---
## 5. Bokeh (Interactive Streaming)

In [ ]:
import openmodelstudio as oms
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource
import numpy as np

viz = oms.create_visualization("signal-plot",
    backend="bokeh",
    description="Real-time signal visualization",
    refresh_interval=10)  # re-render every 10 seconds

x = np.linspace(0, 4 * np.pi, 200)
y = np.sin(x)
source = ColumnDataSource(data=dict(x=x, y=y))

p = figure(title="Signal", width=800, height=400)
p.line("x", "y", source=source, line_width=2, color="#8b5cf6")

output = oms.render(p, viz_id=viz["id"])
oms.publish_visualization(viz["id"])
print("Bokeh visualization published")

---
## 6. NetworkX (Graphs)

In [ ]:
import openmodelstudio as oms
import networkx as nx
import matplotlib.pyplot as plt

viz = oms.create_visualization("model-graph",
    backend="networkx",
    description="Model architecture as a graph")

G = nx.karate_club_graph()
fig, ax = plt.subplots(figsize=(10, 8))
pos = nx.spring_layout(G, seed=42)
nx.draw_networkx(G, pos, ax=ax, node_color="#8b5cf6",
                 edge_color=(0.78, 0.78, 0.78, 0.3),
                 font_color="black", node_size=300)

output = oms.render(fig, viz_id=viz["id"])
oms.publish_visualization(viz["id"])
print("NetworkX graph published")

---
## 7. Datashader (Large Datasets)

In [ ]:
import openmodelstudio as oms
import datashader as ds
import pandas as pd
import numpy as np

viz = oms.create_visualization("embedding-scatter",
    backend="datashader",
    description="1M point embedding visualization")

n = 1_000_000
data = pd.DataFrame({"x": np.random.randn(n), "y": np.random.randn(n)})
canvas = ds.Canvas(plot_width=800, plot_height=600)
agg = canvas.points(data, "x", "y")
img = ds.tf.shade(agg, cmap=["#000000", "#8b5cf6", "#ffffff"])

output = oms.render(img, viz_id=viz["id"])
oms.publish_visualization(viz["id"])
print("Datashader visualization published")

---
## 8. GeoPandas (Maps)

In [ ]:
import openmodelstudio as oms
import geopandas as gpd
import matplotlib.pyplot as plt

viz = oms.create_visualization("data-coverage",
    backend="geopandas",
    description="Geographic data distribution")

url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)
fig, ax = plt.subplots(figsize=(12, 6))
world.plot(ax=ax, color="#8b5cf6", edgecolor=(1, 1, 1, 0.3))
ax.set_title("Data Coverage")

output = oms.render(fig, viz_id=viz["id"])
oms.publish_visualization(viz["id"])
print("GeoPandas map published")

---
## The `render()` Function

`oms.render()` auto-detects the backend from the object type:

| Input Object | Backend | Output |
|-------------|---------|--------|
| `matplotlib.figure.Figure` | matplotlib | SVG |
| `plotly.graph_objects.Figure` | plotly | Plotly JSON |
| `bokeh.model.Model` | bokeh | Bokeh JSON |
| `altair.Chart` | altair | Vega-Lite JSON |
| `plotnine.ggplot` | plotnine | SVG |
| `datashader Image` | datashader | Base64 PNG |
| `networkx.Graph` | networkx | SVG (via matplotlib) |
| `geopandas.GeoDataFrame` | geopandas | SVG (via matplotlib) |

Pass `viz_id=` to push the output to the platform:

```python
output = oms.render(fig, viz_id=viz["id"])
```

Without `viz_id`, `render()` returns the output dict locally but doesn't save it.

---
## Dynamic Visualizations

Set `refresh_interval` to create auto-refreshing visualizations:

In [ ]:
import openmodelstudio as oms

viz = oms.create_visualization("live-metrics",
    backend="plotly",
    refresh_interval=5)  # refresh every 5 seconds

print(f"Created live visualization: {viz['id']}")
print("The platform will re-execute the render function every 5 seconds")

---
## Dashboards

Combine multiple visualizations into a single view with drag-and-drop layout.

In [ ]:
import openmodelstudio as oms

# Create a dashboard
dashboard = oms.create_dashboard("Training Monitor",
    description="Real-time training metrics overview")

print(f"Dashboard: {dashboard['id']}")

---
## Dashboard SDK

In [ ]:
import openmodelstudio as oms

# List dashboards
dashboards = oms.list_dashboards()
print(f"Total dashboards: {len(dashboards)}")

# Get a specific dashboard
# dash = oms.get_dashboard(dashboard_id)

# Update layout programmatically
# oms.update_dashboard(dashboard_id,
#     name="Updated Name",
#     layout=[
#         {"visualization_id": "...", "x": 0, "y": 0, "w": 6, "h": 2},
#         {"visualization_id": "...", "x": 6, "y": 0, "w": 6, "h": 2},
#     ])

# Delete a dashboard
# oms.delete_dashboard(dashboard_id)

---

## Tips

- **Start with Plotly or Altair** for interactive charts — they render live in the browser editor
- **Use matplotlib/seaborn** for publication-quality static figures
- **Use datashader** for datasets with more than 100k points
- **Set `refresh_interval > 0`** for live monitoring dashboards
- **Publish visualizations** before adding them to dashboards
- The in-browser editor at `/visualizations/{id}` has a Monaco code editor, live preview (for JSON backends), template insertion, and data/config tabs

For the full API reference, see [Visualizations & Dashboards docs](../docs/VISUALIZATIONS.md).